<a href="https://colab.research.google.com/github/SweetTooth05/Resume-Demo/blob/main/CNN%20%26%20Tabular%20Neural%20Network%20Predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

####Loading Libraries

In [ ]:
#Referencing Supporting Code Base
#@misc{rosenfelderaipytorch2020,
  #author = {Rosenfelder, Markus},
  #title = {Multi-Input Deep Neural Networks with PyTorch-Lightning - Combine Image and Tabular Data},
  #year = {2020},
  #publisher = {rosenfelder.ai},
  #journal = {rosenfelder.ai},
  #howpublished = {\url{https://rosenfelder.ai/multi-input-neural-network-pytorch/}},
#}
#Importing Libraries
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import requests
from io import BytesIO
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from textblob import TextBlob
import matplotlib.pyplot as plt

####Function Definitions

In [ ]:
#loading Data Function
def load_data():
    """Load and preprocess the gumtree listings data."""
    print("Loading and preprocessing data...")
    df = pd.read_excel('gumtree_listings.xlsx')
    df = df.drop(['listing_url'], axis=1)
    return df

#data preprocessing function dependent functions
# Split engine_config and location
def split_engine_config(val):
    """Extract numerical values from engine configuration string."""
    val_str = str(val)

    # Extract cylinders (format: "# cyl" - integer with space)
    cyl_match = re.search(r'(\d+)\s+cyl', val_str, re.IGNORECASE)
    cyl = int(cyl_match.group(1)) if cyl_match else 0

    # Extract volume (format: "#.#L" - float with no space)
    vol_match = re.search(r'(\d+\.?\d*)L', val_str, re.IGNORECASE)
    vol = float(vol_match.group(1)) if vol_match else 0.0

    return pd.Series({'engine_cylinders': cyl, 'engine_volume': vol})

#splitting the location field into suburb and state
def split_location(val):
    parts = str(val).split(',')
    suburb = parts[0].strip() if len(parts) > 0 else ''
    state = parts[1].strip() if len(parts) > 1 else ''
    return pd.Series({'suburb': suburb, 'state': state})

# Sentiment analysis on description to turn it into a numerical data
def get_sentiment(text):
    return TextBlob(str(text)).sentiment.polarity

####################################
#### Data PreProcessing Function####
####################################
def preprocess_data():
    """Preprocess the gumtree listings data."""
    print("Loading and preprocessing data...")
    # Handle missing values
    numerical_cols = ['odometer', 'price']
    for col in numerical_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())

    categorical_cols = ['body_type', 'transmission', 'manufacturer', 'price_conditions', 'seller_type']
    for col in categorical_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna('Unknown')

    text_cols = ['listing_title', 'description']
    for col in text_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna('')

    if df['location'].isnull().sum() > 0:
        df['location'] = df['location'].fillna('Unknown, Unknown')

    if df['engine_config'].isnull().sum() > 0:
        df['engine_config'] = df['engine_config'].fillna('Unknown')

    if df['featured_image'].isnull().sum() > 0:
        df['featured_image'] = df['featured_image'].fillna('')

    # Apply the function and ensure proper data types
    engine_features = df['engine_config'].apply(split_engine_config)
    df = pd.concat([df, engine_features], axis=1)
    df = df.drop(['engine_config'], axis=1)

    # Ensure proper data types
    df['engine_cylinders'] = df['engine_cylinders'].astype(int)
    df['engine_volume'] = df['engine_volume'].astype(float)

    # Print some statistics to verify the extraction
    print(f"Engine cylinders range: {df['engine_cylinders'].min()} - {df['engine_cylinders'].max()}")
    print(f"Engine volume range: {df['engine_volume'].min():.1f}L - {df['engine_volume'].max():.1f}L")
    print(f"Sample engine data:")
    print(df[['engine_cylinders', 'engine_volume']].head(10))

    df = pd.concat([df, df['location'].apply(split_location)], axis=1)
    df = df.drop(['location'], axis=1)

    df['description_sentiment'] = df['description'].apply(get_sentiment)
    df = df.drop(['description'], axis=1)

    # One-hot encode categorical variables
    cat_cols = ['listing_title', 'body_type', 'transmission', 'manufacturer', 'price_conditions', 'seller_type', 'suburb', 'state']
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    # Normalize numerical columns
    num_cols = ['odometer', 'engine_cylinders', 'engine_volume', 'price']
    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])

    # Limit dataset size for faster training
    df = df.head(2000)  # Use first 2000 samples

    print(f"Preprocessed dataset shape: {df.shape}")

    # Create visualizations before returning the data
    print("\n=== CREATING DATA VISUALIZATIONS ===")

    # Create box plots for numerical features
    create_boxplots(df, num_cols)

    # Create correlation matrix
    create_correlation_matrix(df, num_cols)

    return df

####Feature Analysis Functions

In [ ]:
#taking the pre-processed data to find the features that are highly correlated to the target variable to retain for training
def create_boxplots(df, numerical_cols):
    """Create box plots for all numerical columns to visualize distributions and outliers."""
    print("\n=== CREATING BOX PLOTS ===")

    # Calculate number of subplots needed
    n_cols = len(numerical_cols)
    n_rows = (n_cols + 2) // 3  # 3 plots per row

    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
    fig.suptitle('Box Plots of Numerical Features', fontsize=16, y=0.98)

    # Flatten axes array for easier indexing
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for idx, col in enumerate(numerical_cols):
        row = idx // 3
        col_idx = idx % 3

        # Create box plot
        axes[row, col_idx].boxplot(df[col].dropna(), vert=True)
        axes[row, col_idx].set_title(f'{col} Distribution')
        axes[row, col_idx].set_ylabel('Value')

        # Add statistics as text
        stats = df[col].describe()
        stats_text = f'Mean: {stats["mean"]:.2f}\nStd: {stats["std"]:.2f}\nQ1: {stats["25%"]:.2f}\nQ3: {stats["75%"]:.2f}'
        axes[row, col_idx].text(0.02, 0.98, stats_text, transform=axes[row, col_idx].transAxes,
                               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # Hide empty subplots
    for idx in range(len(numerical_cols), n_rows * 3):
        row = idx // 3
        col_idx = idx % 3
        axes[row, col_idx].set_visible(False)

    plt.tight_layout()
    plt.savefig('numerical_features_boxplots.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("Box plots saved as 'numerical_features_boxplots.png'")

def create_correlation_matrix(df, numerical_cols):
    """Create correlation matrix heatmap for numerical columns."""
    print("\n=== CREATING CORRELATION MATRIX ===")

    # Select only numerical columns
    numerical_df = df[numerical_cols].copy()

    # Calculate correlation matrix
    corr_matrix = numerical_df.corr()

    # Create heatmap
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Create upper triangle mask

    # Create heatmap with custom colors
    sns.heatmap(corr_matrix,
                mask=mask,
                annot=True,
                cmap='coolwarm',
                center=0,
                square=True,
                fmt='.3f',
                cbar_kws={"shrink": .8},
                linewidths=0.5)

    plt.title('Correlation Matrix of Numerical Features', fontsize=16, pad=20)
    plt.tight_layout()
    plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Print correlation with target variable (price)
    if 'price' in corr_matrix.columns:
        print("\nCorrelations with Price:")
        price_corr = corr_matrix['price'].sort_values(ascending=False)
        for feature, corr in price_corr.items():
            if feature != 'price':
                print(f"  {feature}: {corr:.3f}")

    print("Correlation matrix saved as 'correlation_matrix.png'")

    return corr_matrix

####Model Training Functions

In [ ]:
#Defining the layers within a convulutional layer in blocks
#allows for greater modularity and implementation of features to reduce issues such as over fitting
def conv_block(input_size, output_size):
    """Convolutional block with batch normalization and max pooling."""
    block = nn.Sequential(
        nn.Conv2d(input_size, output_size, (3, 3), padding=1),
        nn.ReLU(),
        nn.BatchNorm2d(output_size),
        nn.MaxPool2d((2, 2)),
    )
    return block

def train_model(model, train_loader, val_loader, num_epochs=10, device='cpu'):
    """Train the model."""
    print(f"Training on device: {device}")

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=model.lr)

    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_idx, (images, tabular, targets) in enumerate(train_loader):
            images, tabular, targets = images.to(device), tabular.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(images, tabular)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 20 == 0:
                print(f'Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, tabular, targets in val_loader:
                images, tabular, targets = images.to(device), tabular.to(device), targets.to(device)
                outputs = model(images, tabular)
                loss = criterion(outputs, targets)
                val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f'Epoch {epoch+1}/{num_epochs}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

    return train_losses, val_losses

def plot_training_history(train_losses, val_losses):
    """Plot training history."""
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training History')
    plt.legend()
    plt.grid(True)
    plt.savefig('training_history.png')
    plt.show()

####Model Evaluation Functions

In [ ]:
def evaluate_model(model, test_loader, device='cpu'):
    """Evaluate the model."""
    model.eval()
    predictions = []
    actuals = []

    with torch.no_grad():
        for images, tabular, targets in test_loader:
            images, tabular, targets = images.to(device), tabular.to(device), targets.to(device)
            outputs = model(images, tabular)
            predictions.extend(outputs.cpu().numpy().flatten())
            actuals.extend(targets.cpu().numpy().flatten())

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    mae = mean_absolute_error(actuals, predictions)
    mse = mean_squared_error(actuals, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(actuals, predictions)

    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}")

    return mae, mse, rmse, r2

####Class Definitions

In [ ]:
#since the table will be handling both images and tabular data, a data set class is to be created to handle both of these formats
class CarDataset(Dataset):
    """Tabular and Image dataset for car price prediction."""

    def __init__(self, df, image_size=(224, 224)):
        self.df = df
        self.image_size = image_size
        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        row = self.df.iloc[idx]

        # Get target variable
        y = torch.FloatTensor([row['price']])

        # Download image from URL
        image = self.download_image(row['featured_image'])
        image = self.transform(image)

        # Get tabular features (excluding price and image URL)
        tabular_features = row.drop(['price', 'featured_image']).values
        tabular = torch.FloatTensor(tabular_features.astype(float))

        return image, tabular, y

    def download_image(self, url):
        """Download image from URL."""
        if pd.isna(url) or url == '' or url == 'Unknown':
            # Create a gray placeholder image if no URL
            return Image.new('RGB', self.image_size, color='gray')

        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                img = Image.open(BytesIO(resp.content)).convert('RGB')
                return img
            else:
                # Create a gray placeholder image if download fails
                return Image.new('RGB', self.image_size, color='gray')
        except:
            # Create a gray placeholder image if any error occurs
            return Image.new('RGB', self.image_size, color='gray')

#Creating a class for the carprice predicitng model for handling all
class CarPricePredictor(nn.Module):
    def __init__(self, tabular_input_size, lr=1e-3):
        super().__init__()
        self.lr = lr

        # Image processing layers
        self.conv1 = conv_block(3, 16)
        self.conv2 = conv_block(16, 32)
        self.conv3 = conv_block(32, 64)
        self.conv4 = conv_block(64, 128)

        # Calculate the size after convolutions
        # Starting with 224x224, after 4 maxpool layers: 224 -> 112 -> 56 -> 28 -> 14
        self.image_features_size = 128 * 14 * 14

        # Image feature processing
        self.img_fc1 = nn.Linear(self.image_features_size, 256)
        self.img_fc2 = nn.Linear(256, 64)

        # Tabular feature processing
        self.tab_fc1 = nn.Linear(tabular_input_size, 128)
        self.tab_fc2 = nn.Linear(128, 64)
        self.tab_fc3 = nn.Linear(64, 32)

        # Combined processing
        self.combined_fc1 = nn.Linear(64 + 32, 64)
        self.combined_fc2 = nn.Linear(64, 32)
        self.combined_fc3 = nn.Linear(32, 1)

        # Dropout for regularization
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()

    def forward(self, img, tab):
        # Process image
        img = self.conv1(img)
        img = self.conv2(img)
        img = self.conv3(img)
        img = self.conv4(img)
        img = img.view(img.size(0), -1)  # Flatten
        img = self.relu(self.img_fc1(img))
        img = self.dropout(img)
        img = self.relu(self.img_fc2(img))

        # Process tabular data
        tab = self.relu(self.tab_fc1(tab))
        tab = self.dropout(tab)
        tab = self.relu(self.tab_fc2(tab))
        tab = self.relu(self.tab_fc3(tab))

        # Combine features
        combined = torch.cat((img, tab), dim=1)
        combined = self.relu(self.combined_fc1(combined))
        combined = self.dropout(combined)
        combined = self.relu(self.combined_fc2(combined))

        return self.combined_fc3(combined)

####Main Function

Data Preparation

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

#loadign and pre-processing the data
df = load_data()
df = preprocess_data(df)

# Create dataset class
dataset = CarDataset(df)

# Split dataset
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
#using the pytorch module rnsom split for training, validation and testing data sets
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

Using device: cuda
Loading and preprocessing data...
Preprocessed dataset shape: (2000, 14158)


Data Processing

In [ ]:
# Create data loaders for the pytorch model
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Get tabular input size
sample_row = df.iloc[0]
tabular_input_size = len(sample_row.drop(['price', 'featured_image']))

Model Creation and Training

In [ ]:
# Create model
model = CarPricePredictor(tabular_input_size=tabular_input_size, lr=1e-3)
model = model.to(device)

print(f"Model created with tabular input size: {tabular_input_size}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")

# Train model
train_losses, val_losses = train_model(model, train_loader, val_loader, num_epochs=15, device=device)

# Plot training history
plot_training_history(train_losses, val_losses)

Model created with tabular input size: 14156
Total parameters: 8367905
Training on device: cuda
Epoch 1/15, Batch 0/88, Loss: 0.0291
Epoch 1/15, Batch 20/88, Loss: 0.0096
Epoch 1/15, Batch 40/88, Loss: 0.0307
Epoch 1/15, Batch 60/88, Loss: 0.0086
Epoch 1/15, Batch 80/88, Loss: 0.0042
Epoch 1/15: Train Loss: 0.0110, Val Loss: 0.0024
Epoch 2/15, Batch 0/88, Loss: 0.0029
Epoch 2/15, Batch 20/88, Loss: 0.0018
Epoch 2/15, Batch 40/88, Loss: 0.0009
Epoch 2/15, Batch 60/88, Loss: 0.0014
Epoch 2/15, Batch 80/88, Loss: 0.0025
Epoch 2/15: Train Loss: 0.0017, Val Loss: 0.0007
Epoch 3/15, Batch 0/88, Loss: 0.0009
Epoch 3/15, Batch 20/88, Loss: 0.0038
Epoch 3/15, Batch 40/88, Loss: 0.0005
Epoch 3/15, Batch 60/88, Loss: 0.0002
Epoch 3/15, Batch 80/88, Loss: 0.0009
Epoch 3/15: Train Loss: 0.0009, Val Loss: 0.0005
Epoch 4/15, Batch 0/88, Loss: 0.0003
Epoch 4/15, Batch 20/88, Loss: 0.0002
Epoch 4/15, Batch 40/88, Loss: 0.0002
Epoch 4/15, Batch 60/88, Loss: 0.0002
Epoch 4/15, Batch 80/88, Loss: 0.0017
E

Model Evaluation

In [ ]:
# Evaluate model
print("\nModel Evaluation:")
evaluate_model(model, test_loader, device=device)